# kld_sweep.py — Linux / Colab test
Tests `kld_sweep.py` on **Qwen3.5-0.8B** using two small quants from unsloth.
- BF16: 1.52 GiB
- Quants: UD-Q4_K_XL (559 MB) + UD-IQ2_XXS (338 MB)
- Total disk: ~2.5 GiB

**Runtime:** GPU (T4 recommended). CPU-only will work but logits generation will be slow.

**Steps:** Run all cells in order. The sweep resumes automatically if interrupted.

In [ ]:
# @title 1 — Install dependencies
!pip install pandas matplotlib adjustText scipy -q

In [ ]:
# @title 2 — Download llama.cpp (latest release, Linux CUDA build)
import urllib.request, json, zipfile, os, glob

api = urllib.request.urlopen('https://api.github.com/repos/ggerganov/llama.cpp/releases/latest')
release = json.loads(api.read())
assets = release['assets']

# Pick linux + cuda build
target = next(
    a for a in assets
    if 'linux' in a['name'].lower() and 'cuda' in a['name'].lower() and a['name'].endswith('.zip')
)
print(f"Downloading: {target['name']} ({target['size'] / 1024**2:.1f} MB)")
urllib.request.urlretrieve(target['browser_download_url'], 'llama.zip')

with zipfile.ZipFile('llama.zip') as z:
    z.extractall('llama_bin')

perplexity = glob.glob('llama_bin/**/llama-perplexity', recursive=True)
assert perplexity, 'llama-perplexity not found in archive'
PPL_EXE = perplexity[0]
os.chmod(PPL_EXE, 0o755)
print(f'llama-perplexity: {PPL_EXE}')

In [ ]:
# @title 3 — Download BF16 model and quants from HuggingFace
!pip install huggingface_hub -q
from huggingface_hub import hf_hub_download
import os

os.makedirs('bf16',   exist_ok=True)
os.makedirs('quants', exist_ok=True)

REPO = 'unsloth/Qwen3.5-0.8B-GGUF'

bf16_path = hf_hub_download(REPO, 'Qwen3.5-0.8B-BF16.gguf',     local_dir='bf16')
q1_path   = hf_hub_download(REPO, 'Qwen3.5-0.8B-UD-Q4_K_XL.gguf', local_dir='quants')
q2_path   = hf_hub_download(REPO, 'Qwen3.5-0.8B-UD-IQ2_XXS.gguf', local_dir='quants')

print(f'BF16  : {bf16_path}')
print(f'Quant1: {q1_path}')
print(f'Quant2: {q2_path}')

In [ ]:
# @title 4 — Download kld_sweep.py and dataset
# Replace the URLs below with your actual hosted versions if needed
# (e.g. raw GitHub URL once you publish the repo)

# Dataset — using wikitext2 test set as a neutral default
# You can replace with your own dataset
!wget -q https://raw.githubusercontent.com/openai/gpt-2/master/src/encode.py -O /dev/null  # connectivity check

import urllib.request
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/ggerganov/llama.cpp/master/scripts/get-wikitext-2.sh',
    'get-wikitext-2.sh'
)
!bash get-wikitext-2.sh
DATASET = 'wikitext-2-raw/wiki.test.raw'
print(f'Dataset: {DATASET}')

# Upload kld_sweep.py manually or fetch from your repo
# For now, upload it via the Colab file browser and set the path below
SCRIPT = 'kld_sweep.py'  # adjust if needed
print(f'Script : {SCRIPT}')

In [ ]:
# @title 5 — Run sweep
import subprocess, sys

cmd = [
    sys.executable, SCRIPT,
    '--exe',        PPL_EXE,
    '--bf16',       bf16_path,
    '--quants',     'quants',
    '--dataset',    DATASET,
    '--output',     'output',
    '--model-name', 'Qwen3.5-0.8B',
    '--args',       '-t 4 -c 512 -ngl 99',
]

print('Command:', ' '.join(cmd))
result = subprocess.run(cmd)
print('\nExit code:', result.returncode)

In [ ]:
# @title 6 — Show results
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv('output/Qwen3.5-0.8B_results.csv')
display(df.sort_values('KLD_Score'))

display(Image('output/kld_plot_Qwen3.5-0.8B.png'))
display(Image('output/ppl_plot_Qwen3.5-0.8B.png'))